# TechNSports — Health Map Production Pipeline

**What this notebook does:**  
1. Loads the reference population models (built from NHANES data)  
2. Reads the client's InBody CSV + ShapeScale sheet entry  
3. Merges both scans into a unified 46-variable record  
4. Projects the client onto the Health Map  
5. Generates all figures and saves them to Google Drive  

**For Stephanie:** Fill in the cells marked `📝 FILL IN` and then click **Runtime → Run all**.

---

## Step 0 — Setup (run once per session)

In [ ]:
# Install required packages (takes ~30 seconds on first run)
import subprocess, sys
packages = ["scikit-learn", "pandas", "numpy", "matplotlib", "scipy", "pyreadstat"]
for pkg in packages:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
print("✅ Packages ready")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print("✅ Google Drive mounted at /content/drive")

In [ ]:
import sys, os
from pathlib import Path

# ── Drive root ───────────────────────────────────────────────────────────────
# Adjust if your Shared Drive name differs
DRIVE_ROOT = Path("/content/drive/Shareddrives/TechNSports Project Hub")
if not DRIVE_ROOT.exists():
    # Fallback: personal My Drive
    DRIVE_ROOT = Path("/content/drive/MyDrive/TechNSports Project Hub")

PIPELINE_DIR = DRIVE_ROOT / "02_MEXICO" / "PCA_Pipeline"
NHANES_DIR   = DRIVE_ROOT / "02_MEXICO" / "NHANES_RAW"
MODEL_DIR    = NHANES_DIR / "models"
DATA_MASTER  = DRIVE_ROOT / "02_MEXICO" / "TNS_Scan_Data_Master.xlsx"

# Add pipeline dir to Python path
sys.path.insert(0, str(PIPELINE_DIR))

# Verify paths
for label, p in [("Pipeline", PIPELINE_DIR), ("NHANES", NHANES_DIR), ("Models", MODEL_DIR)]:
    status = "✅" if p.exists() else "❌ NOT FOUND"
    print(f"{status} {label}: {p}")

In [ ]:
# Import TNS modules
from tns_inbody_parser   import parse_inbody_csv, parse_inbody_csv_string, inbody_summary
from tns_shapescale_reader import parse_shapescale_csv, parse_shapescale_sheet, shapescale_summary
from tns_reconcile       import reconcile_scanners, reconcile_summary
from tns_pca_pipeline    import load_all_models, project_client, validate_model
from tns_visualize       import generate_client_figures
import pandas as pd, json

# Load all PCA models
MODELS = load_all_models(MODEL_DIR)
if MODELS:
    print(f"✅ Loaded {len(MODELS)} lens model(s): {list(MODELS.keys())}")
else:
    print("❌ No models found. Run Step 1 (Build Models) first.")

---
## Step 1 — Build Reference Population Models

> **Run this only once** (or when you want to rebuild with updated NHANES data).  
> Takes ~3–5 minutes. Skip to Step 2 if models already exist.

Prerequisite: NHANES XPT files must be in `NHANES_RAW/2017-2018/` and `NHANES_RAW/2015-2016/`.

In [ ]:
# ── Only run this cell when you need to (re)build the models ──────────────────
BUILD_MODELS = False   # 📝 Change to True to build / rebuild

if BUILD_MODELS:
    from tns_pca_pipeline import build_all_models
    saved = build_all_models(
        nhanes_dir = NHANES_DIR,
        out_dir    = MODEL_DIR,
    )
    print(f"\n✅ Built {len(saved)} models:")
    for lens, path in saved.items():
        print(f"   {lens}: {path.name}")
    # Reload models
    MODELS = load_all_models(MODEL_DIR)
else:
    print("Skipping model build (BUILD_MODELS = False)")

In [ ]:
# Optional: validate models (prints variance explained, top loadings)
VALIDATE = False   # set True to inspect
if VALIDATE and MODELS:
    for lens_name, model in MODELS.items():
        validate_model(model)

---
## Step 2 — 📝 Enter Client Information

Fill in the fields below, then continue.

In [ ]:
# ── 📝 FILL IN: Client details ────────────────────────────────────────────────
CLIENT_ID    = "garcia_jesus"         # snake_case unique ID (used for file names)
CLIENT_NAME  = "Jesus Garcia"         # Display name for figures
CLIENT_SEX   = "M"                    # "M" or "F"
HEIGHT_CM    = 179.0                  # Standing height in cm from intake form
SCAN_LABEL   = "intake"               # "intake" | "8wk" | "16wk" | "24wk"
LENS         = "auto"                 # "auto" or specific: "health" | "body_comp" etc.

# Output folder (auto-created)
CLIENT_OUTPUT_DIR = MODEL_DIR.parent / "client_projections" / CLIENT_ID
CLIENT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Client : {CLIENT_NAME} ({CLIENT_ID})")
print(f"Height : {HEIGHT_CM} cm | Sex: {CLIENT_SEX} | Visit: {SCAN_LABEL}")
print(f"Output : {CLIENT_OUTPUT_DIR}")

---
## Step 3 — Load InBody Scan

**Option A** (recommended): Upload the CSV file exported from LookinBody Cloud.  
**Option B**: Paste the CSV text directly into the cell below.

In [ ]:
# ── Option A: Upload InBody CSV ───────────────────────────────────────────────
INBODY_FROM_UPLOAD = False   # 📝 Set to True if uploading a file

IB_SCANS = []
if INBODY_FROM_UPLOAD:
    from google.colab import files
    uploaded = files.upload()
    for fname in uploaded.keys():
        IB_SCANS = parse_inbody_csv(fname)
        print(f"Loaded {len(IB_SCANS)} scan(s) from {fname}")

In [ ]:
# ── Option B: Paste InBody CSV text ──────────────────────────────────────────
# Copy the entire CSV from LookinBody Cloud (including the header row)
# and paste between the triple quotes below.

INBODY_CSV_TEXT = """\
Date,Measurement device.,Weight(lb),Skeletal Muscle Mass(lb),Soft Lean Mass(lb),Body Fat Mass(lb),BMI(kg/m²),Percent Body Fat(%),Basal Metabolic Rate(kJ),InBody Score,Right Arm Lean Mass(lb),Left Arm Lean Mass(lb),Trunk Lean Mass(lb),Right Leg Lean Mass(lb),Left leg Lean Mass(lb),Right Arm Fat Mass(lb),Left Arm Fat Mass(lb),Trunk Fat Mass(lb),Right Leg Fat Mass(lb),Left Leg Fat Mass(lb),Right Arm ECW Ratio,Left Arm ECW Ratio,Trunk ECW Ratio,Right Leg ECW Ratio,Left Leg ECW Ratio,Waist Hip Ratio,Waist Circumference(cm),Visceral Fat Area(cm²),Visceral Fat Level(Level),Total Body Water(lb),Intracellular Water(lb),Extracellular Water(lb),ECW Ratio,Upper-Lower,Upper,Lower,Leg Muscle Level(Level),Leg Lean Mass(lb),Protein(lb),Mineral(lb),Bone Mineral Content(lb),Body Cell Mass(lb),SMI(kg/m²),Whole Body Phase Angle(°)
20260412153252,270S,219.1,88.0,-,65.3,34.4,29.8,7853,78.0,9.11,9.19,68.1,22.00,21.72,4.4,4.4,35.9,8.6,8.6,-,-,-,-,-,0.95,-,-,12.0,112.9,-,-,-,1,0,0,-,-,30.6,10.45,-,-,9.7,6.9
"""

# Parse (only runs if Option A didn't load scans)
if not IB_SCANS and INBODY_CSV_TEXT.strip():
    IB_SCANS = parse_inbody_csv_string(INBODY_CSV_TEXT)

if IB_SCANS:
    print(f"✅ InBody: {len(IB_SCANS)} scan(s) loaded")
    for s in IB_SCANS:
        print(" ", inbody_summary(s))
else:
    print("❌ No InBody scans loaded. Check the CSV text above.")

---
## Step 3b — 📝 Enter Lab Values (optional)

Fill in values from the **Lab Entry** sheet or from the printed lab report.  
Leave as `None` if labs are not yet available — the pipeline will continue with scan-only mode and show a **Partial Data** badge on affected lenses.

> Units: mg/dL for lipids and glucose · % for HbA1c · µIU/mL for insulin · mg/L for hs-CRP · mmHg for blood pressure

In [ ]:
# ── 📝 FILL IN: Lab values from Lab Entry sheet or printed report ─────────────

LAB_DATA = {
    "lab_total_chol":    None,   # 📝 mg/dL  total cholesterol  (range 50–500)
    "lab_hdl":           None,   # 📝 mg/dL  HDL               (range 10–200)
    "lab_ldl":           None,   # 📝 mg/dL  LDL               (range 20–400)
    "lab_triglycerides": None,   # 📝 mg/dL  triglycerides     (range 20–2000)
    "lab_glucose":       None,   # 📝 mg/dL  fasting glucose    (range 30–500)
    "lab_hba1c":         None,   # 📝 %      HbA1c             (range 3–18)
    "lab_insulin":       None,   # 📝 µIU/mL fasting insulin   (range 0–300)
    "lab_hscrp":         None,   # 📝 mg/L   hs-CRP            (range 0.01–100)
    "lab_sbp":           None,   # 📝 mmHg   systolic BP       (range 70–250)
    "lab_dbp":           None,   # 📝 mmHg   diastolic BP      (range 40–150)
}

# ── OR: Auto-read from Lab Entry sheet in TNS_Scan_Data_Master.xlsx ──────────
READ_LABS_FROM_XLSX = False   # 📝 Set to True to auto-read

if READ_LABS_FROM_XLSX and DATA_MASTER.exists():
    df_lab = pd.read_excel(DATA_MASTER, sheet_name="Lab Entry", dtype=str).fillna("")
    client_labs = df_lab[df_lab["client_id"].str.lower().str.strip() == CLIENT_ID.lower()]
    if not client_labs.empty:
        row = client_labs.iloc[-1]  # most recent row
        def _sf(v):
            try: return float(v) if str(v).strip() not in ("", "-") else None
            except: return None
        LAB_DATA = {
            "lab_total_chol":    _sf(row.get("lab_total_chol", "")),
            "lab_hdl":           _sf(row.get("HDL", "")),
            "lab_ldl":           _sf(row.get("LDL", "")),
            "lab_triglycerides": _sf(row.get("lab_triglycerides", "")),
            "lab_glucose":       _sf(row.get("lab_glucose", "")),
            "lab_hba1c":         _sf(row.get("lab_HbA1c", row.get("lab_hba1c", ""))),
            "lab_insulin":       _sf(row.get("lab_insulin", "")),
            "lab_hscrp":         _sf(row.get("lab_hs-CRP", row.get("lab_hscrp", ""))),
            "lab_sbp":           _sf(row.get("lab_SBP", "")),
            "lab_dbp":           _sf(row.get("lab_DBP", "")),
        }
        print("✅ Labs loaded from XLSX")
    else:
        print("⚠️  No lab rows found for this client in Lab Entry sheet")

provided_labs = [k for k, v in LAB_DATA.items() if v is not None]
if provided_labs:
    print(f"✅ Labs: {len(provided_labs)}/10 values provided: {provided_labs}")
else:
    print("ℹ️  Labs: none provided — scan-only mode. Health lens will use population medians for lab variables.")

---
## Step 3c — 📝 Enter Lifestyle Data (optional)

From the intake form or **Lifestyle Entry** sheet.  
Physical activity values improve accuracy of the Performance lens.  
Sleep, stress, and subjective health feed the Longevity lens.

In [ ]:
# ── 📝 FILL IN: Lifestyle values from intake form or Lifestyle Entry sheet ────

LIFESTYLE_DATA = {
    "lifestyle_vig_min_week":        None,  # 📝 min/week vigorous activity  (e.g. 60)
    "lifestyle_mod_min_week":        None,  # 📝 min/week moderate activity  (e.g. 150)
    "lifestyle_sed_hours_day":       None,  # 📝 hrs/day sedentary           (e.g. 8.5)
    "lifestyle_sleep_hours":         None,  # 📝 hrs/night sleep             (e.g. 7.0)
    "lifestyle_smoker_score":        None,  # 📝 0=Never  1=Former  2=Current
    "lifestyle_alcohol_drinks_week": None,  # 📝 standard drinks/week        (e.g. 4)
    "lifestyle_stress_score":        None,  # 📝 self-rated stress 1–10      (e.g. 6)
    "lifestyle_subj_health_score":   None,  # 📝 self-rated health 1–10      (e.g. 7)
}

# ── OR: Auto-read from Lifestyle Entry sheet ──────────────────────────────────
READ_LIFESTYLE_FROM_XLSX = False   # 📝 Set to True to auto-read

if READ_LIFESTYLE_FROM_XLSX and DATA_MASTER.exists():
    df_life = pd.read_excel(DATA_MASTER, sheet_name="Lifestyle Entry", dtype=str).fillna("")
    client_life = df_life[df_life["client_id"].str.lower().str.strip() == CLIENT_ID.lower()]
    if not client_life.empty:
        row = client_life.iloc[-1]
        def _sf(v):
            try: return float(v) if str(v).strip() not in ("", "-") else None
            except: return None
        smoker_map = {"never": 0, "former": 1, "current": 2}
        # Column headers in the sheet have newlines — normalise
        row_flat = {k.replace("\n", " "): v for k, v in row.items()}
        LIFESTYLE_DATA = {
            "lifestyle_vig_min_week":        _sf(row_flat.get("vigorous_min per_week", "")),
            "lifestyle_mod_min_week":        _sf(row_flat.get("moderate_min per_week", "")),
            "lifestyle_sed_hours_day":       _sf(row_flat.get("sedentary_hrs per_day", "")),
            "lifestyle_sleep_hours":         _sf(row_flat.get("sleep_hrs per_night", "")),
            "lifestyle_smoker_score":        smoker_map.get(
                                                str(row_flat.get("smoker", "")).strip().lower()),
            "lifestyle_alcohol_drinks_week": _sf(row_flat.get("alcohol_drinks per_week", "")),
            "lifestyle_stress_score":        _sf(row_flat.get("stress (1-10)", "")),
            "lifestyle_subj_health_score":   _sf(row_flat.get("subj_health (1-10)", "")),
        }
        print("✅ Lifestyle loaded from XLSX")
    else:
        print("⚠️  No lifestyle rows found for this client")

provided_life = [k for k, v in LIFESTYLE_DATA.items() if v is not None]
if provided_life:
    print(f"✅ Lifestyle: {len(provided_life)}/8 values provided: {provided_life}")
else:
    print("ℹ️  Lifestyle: none provided — performance/longevity lenses will use population medians.")

---
## Step 4 — Load ShapeScale Scan

The ShapeScale data comes from the **ShapeScale Entry** sheet in `TNS_Scan_Data_Master.xlsx`.  
Option A reads the XLSX directly from Drive. Option B lets you paste manually.

In [ ]:
# ── Option A: Read from TNS_Scan_Data_Master.xlsx ─────────────────────────────
SS_SCANS = []
SHAPESCALE_FROM_XLSX = True   # 📝 Set to False if you prefer to paste manually

if SHAPESCALE_FROM_XLSX and DATA_MASTER.exists():
    df_ss = pd.read_excel(DATA_MASTER, sheet_name="ShapeScale Entry", dtype=str)
    # Filter to current client
    if "client_id" in df_ss.columns:
        df_client = df_ss[df_ss["client_id"].str.lower().str.strip() == CLIENT_ID.lower()]
    else:
        df_client = df_ss   # take all rows if no client_id column
    rows = df_client.fillna("").to_dict(orient="records")
    SS_SCANS = parse_shapescale_sheet(rows)
    print(f"✅ ShapeScale: {len(SS_SCANS)} scan(s) loaded from XLSX")
    for s in SS_SCANS:
        print(" ", shapescale_summary(s))
elif SHAPESCALE_FROM_XLSX:
    print(f"❌ XLSX not found at {DATA_MASTER}. Try Option B below.")

In [ ]:
# ── Option B: Manual ShapeScale entry ────────────────────────────────────────
# Only needed if XLSX read failed. Fill in values from the ShapeScale app/PDF.

SHAPESCALE_MANUAL = not bool(SS_SCANS)   # auto-enable if Option A didn't load

if SHAPESCALE_MANUAL:
    manual_ss_row = {
        "client_id":           CLIENT_ID,
        "scan_date":           "2026-04-13",    # 📝 YYYY-MM-DD
        "weight_lb":           "215.3",          # 📝
        "body_fat_pct":        "34.5",           # 📝
        "lean_mass_lb":        "140.9",          # 📝
        "bmi":                 "33.8",           # 📝
        "bmr_cal":             "1685",           # 📝
        "shape_score":         "72",             # 📝
        "health_score":        "65",             # 📝
        "body_age":            "46",             # 📝
        "visceral_fat_rating": "Poor",           # 📝 Very Poor/Poor/Fair/Good/Excellent
        # Circumferences (inches) ──────────────────────────────────
        "neck_in":             "16.5",           # 📝
        "shoulders_in":        "51.2",           # 📝
        "chest_in":            "44.1",           # 📝
        "waist_in":            "44.0",           # 📝
        "hips_in":             "44.0",           # 📝
        "bicep_l_in":          "14.8",           # 📝
        "bicep_r_in":          "15.0",           # 📝
        "thigh_l_in":          "25.1",           # 📝
        "thigh_r_in":          "25.3",           # 📝
        "calf_l_in":           "15.9",           # 📝
        "calf_r_in":           "16.1",           # 📝
        # Pre-computed ratios ──────────────────────────────────────
        "whr":                 "1.00",           # 📝
        "whtr":                "0.564",          # 📝
        "shoulder_waist":      "1.165",          # 📝
    }
    SS_SCANS = parse_shapescale_sheet([manual_ss_row])
    print(f"✅ ShapeScale (manual): {len(SS_SCANS)} scan(s) loaded")
    for s in SS_SCANS:
        print(" ", shapescale_summary(s))

---
## Step 5 — Reconcile Scans & Add Labs (optional)

In [ ]:
# ── 📝 OPTIONAL: Add lab values if available from referral ───────────────────
# Leave as None if not yet available — they'll be imputed from population medians.

EXTRA_LABS = {
    "lab_total_chol":    None,   # mg/dL
    "lab_hdl":           None,   # mg/dL
    "lab_ldl":           None,   # mg/dL
    "lab_triglycerides": None,   # mg/dL
    "lab_glucose":       None,   # mg/dL (fasting)
    "lab_hba1c":         None,   # %
    "lab_insulin":       None,   # μU/mL
    "lab_hscrp":         None,   # mg/L
    "lab_sbp":           None,   # mmHg
    "lab_dbp":           None,   # mmHg
}

In [ ]:
# Reconcile: merge InBody + ShapeScale + labs + lifestyle into unified record
if not IB_SCANS:
    raise RuntimeError("❌ No InBody data. Complete Step 3 first.")
if not SS_SCANS:
    raise RuntimeError("❌ No ShapeScale data. Complete Step 4 first.")

# Use most recent scan of each type
ib_scan = sorted(IB_SCANS, key=lambda s: s.get("scan_date", ""))[-1]
ss_scan = sorted(SS_SCANS, key=lambda s: s.get("ss_scan_date", ""))[-1]

UNIFIED = reconcile_scanners(
    inbody_data    = ib_scan,
    shapescale_data= ss_scan,
    height_cm      = HEIGHT_CM,
    client_id      = CLIENT_ID,
    scan_label     = SCAN_LABEL,
    extra_labs     = LAB_DATA if 'LAB_DATA' in dir() else None,
    lifestyle_data = LIFESTYLE_DATA if 'LIFESTYLE_DATA' in dir() else None,
)

print("✅", reconcile_summary(UNIFIED))

dc = UNIFIED.get("data_completeness", {})
badge = "🟢 Full Data" if dc.get("full_data") else "🟡 Partial Data"
print(f"\n{badge}")
print(f"  Scan: {'✓' if dc.get('scan') else '✗'}  "
      f"Labs: {'✓' if dc.get('labs') else '✗'}  "
      f"Lifestyle: {'✓' if dc.get('lifestyle') else '✗'}")

if UNIFIED.get("flags"):
    print("\n⚠️  Cross-scanner flags:")
    for f in UNIFIED["flags"]:
        print(" •", f)

---
## Step 6 — Project Client onto Health Map

In [ ]:
# ── 📝 OPTIONAL: Load prior projections for trajectory ────────────────────────
# Add previous visit results here to get the trajectory arrow and timeline chart.
# Leave as empty list if this is the first scan.

PRIOR_FILE = CLIENT_OUTPUT_DIR / "projections_history.json"
PRIOR_PROJECTIONS = []

if PRIOR_FILE.exists():
    with open(PRIOR_FILE) as f:
        PRIOR_PROJECTIONS = json.load(f)
    print(f"✅ Loaded {len(PRIOR_PROJECTIONS)} prior projection(s)")
else:
    print("No prior projections found — this is a first scan.")

In [ ]:
# Run the projection
RESULT = project_client(
    client_data          = UNIFIED,
    models               = MODELS,
    lens                 = LENS,
    previous_projections = PRIOR_PROJECTIONS,
    lab_data             = LAB_DATA if 'LAB_DATA' in dir() else None,
    lifestyle_data       = LIFESTYLE_DATA if 'LIFESTYLE_DATA' in dir() else None,
)

dc = RESULT.get("data_completeness", {})
badge = "🟢 Full Data" if dc.get("full_data") else "🟡 Partial Data"

print(f"\n{'='*55}")
print(f"  {CLIENT_NAME} — {SCAN_LABEL.upper()}  {badge}")
print(f"{'='*55}")
print(f"  Lens      : {RESULT['lens_used']} — {RESULT['lens_description']}")
print(f"  PC1/PC2   : {RESULT['pc1']:.4f} / {RESULT['pc2']:.4f}")
print(f"  Percentile: {RESULT['percentile_pc1']:.1f}th (vs NHANES reference)")
print(f"  Coverage  : {RESULT['n_vars_provided']}/{RESULT['n_vars_total']} vars  "
      f"({dc.get('provided_pct', 0):.0f}%)")
if RESULT.get("imputed_vars"):
    print(f"  Imputed   : {RESULT['imputed_vars']}")
print(f"  Top drivers: {RESULT['top_drivers']}")
if RESULT.get('trajectory'):
    t = RESULT['trajectory']
    sign = '+' if t['delta_percentile'] > 0 else ''
    emoji = '📈' if t['direction'] == 'improved' else '📉' if t['direction'] == 'declined' else '➡️'
    print(f"  Trajectory : {emoji} {t['direction'].upper()} | "
          f"Δ = {sign}{t['delta_percentile']:.1f} pct pts since {t['prior_label']}")
print(f"{'='*55}")

In [ ]:
# Save projection to history (appends to projections_history.json)
record_to_save = {
    "pc1":             RESULT["pc1"],
    "pc2":             RESULT["pc2"],
    "percentile_pc1":  RESULT["percentile_pc1"],
    "lens_used":       RESULT["lens_used"],
    "date":            ib_scan.get("scan_date") or ss_scan.get("ss_scan_date", "unknown"),
    "label":           SCAN_LABEL,
    "n_vars_provided": RESULT["n_vars_provided"],
}
PRIOR_PROJECTIONS.append(record_to_save)
with open(PRIOR_FILE, "w") as f:
    json.dump(PRIOR_PROJECTIONS, f, indent=2)
print(f"✅ Saved projection to {PRIOR_FILE.name}")

---
## Step 7 — Generate Figures

In [ ]:
# Determine which model to pass (matches the lens that was actually used)
MODEL_USED = MODELS[RESULT["lens_used"]]

# Prior projections for trajectory (exclude the one we just saved)
priors_for_plot = [p for p in PRIOR_PROJECTIONS if p.get("label") != SCAN_LABEL]

SAVED_FIGS = generate_client_figures(
    projection_result    = RESULT,
    model                = MODEL_USED,
    client_name          = CLIENT_NAME,
    save_dir             = CLIENT_OUTPUT_DIR,
    previous_projections = priors_for_plot if priors_for_plot else None,
    current_unified      = UNIFIED,
    baseline_unified     = None,   # set to intake UNIFIED dict if available
    sex                  = CLIENT_SEX,
    mode                 = "simple",   # "simple" for client, "technical" for clinician
)

print(f"\n✅ Figures saved to: {CLIENT_OUTPUT_DIR}")
for name, path in SAVED_FIGS.items():
    print(f"   {name}: {path.name}")

In [ ]:
# Display figures inline for review
from IPython.display import Image, display
for name, path in SAVED_FIGS.items():
    print(f"\n── {name.replace('_', ' ').title()} ──")
    display(Image(str(path), width=700))

---
## Step 8 — Export Summary Report (JSON)

Saves a machine-readable report for the Advanced Insight Report pipeline.

In [ ]:
import datetime

report = {
    "client_id":     CLIENT_ID,
    "client_name":   CLIENT_NAME,
    "scan_label":    SCAN_LABEL,
    "generated_at":  datetime.datetime.now().isoformat(),
    "projection":    RESULT,
    "scan_dates": {
        "inbody":    ib_scan.get("scan_date"),
        "shapescale": ss_scan.get("ss_scan_date"),
    },
    "key_metrics": {
        "bf_pct":            UNIFIED.get("bf_pct"),
        "smm_kg":            UNIFIED.get("ib_smm_kg"),
        "waist_cm":          UNIFIED.get("ss_waist_cm"),
        "whr":               UNIFIED.get("whr"),
        "bmi":               UNIFIED.get("bmi"),
        "ffmi":              UNIFIED.get("ffmi"),
        "phase_angle":       UNIFIED.get("ib_phase_angle"),
        "inbody_score":      UNIFIED.get("ib_score"),
        "shape_score":       UNIFIED.get("ss_shape_score"),
        "visceral_fat_level":UNIFIED.get("ib_visceral_fat_level"),
    },
    "flags":     UNIFIED.get("flags", []),
    "figure_paths": {k: str(v) for k, v in SAVED_FIGS.items()},
}

report_path = CLIENT_OUTPUT_DIR / f"{CLIENT_ID}_{SCAN_LABEL}_report.json"
with open(report_path, "w") as f:
    json.dump(report, f, indent=2, default=str)

print(f"✅ Report saved: {report_path.name}")
print(f"\n── Key Metrics ──────────────────────────")
for k, v in report["key_metrics"].items():
    if v is not None:
        print(f"  {k:25s}: {v}")

---
## Done!

Figures and report are saved to:  
`02_MEXICO/NHANES_RAW/client_projections/<client_id>/`

Ready to attach to the Advanced Insight Report.

---
*TechNSports Health Intelligence · Merida, Yucatan, Mexico*

---
## Step 9 — Run Tests (optional)

Runs the integration test suite against the three fixture files.  
No NHANES files required — tests use stub models and complete in a few seconds.

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_project_client.py", "-v", "--tb=short"],
    capture_output=True,
    text=True,
    cwd=str(PIPELINE_DIR),
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)